In [1]:
import torch
from torchvision.models import mobilenet_v3_large as mobilenet
from torchvision.models import MobileNet_V3_Large_Weights as pre_weights

from sklearn.metrics import confusion_matrix, accuracy_score

import helpers.data_prefetcher as data_prefetcher
import helpers.data_handler as data_handler
import helpers.utils as utils

### System details

In [2]:
print(f"PyTorch version: {torch.__version__}")

print("--------------------------------------------------")
print(f"Using cuda: {torch.cuda.is_available()}")
print(f"Cuda corrent device: {torch.cuda.current_device()}")
print(f"Cuda device: {torch.cuda.get_device_name(torch.cuda.current_device())}")
print(f"Torch Backend enable: {torch.backends.cudnn.enabled}")
print(f"Torch Backend: {torch.backends.cudnn.version() }")

PyTorch version: 2.2.1+cu121
--------------------------------------------------
Using cuda: True
Cuda corrent device: 0
Cuda device: NVIDIA GeForce GTX 1660 SUPER
Torch Backend enable: True
Torch Backend: 8902


### Getting data

In [3]:
NUM_CLASSES = 1

train, validation, test = data_handler.get_datasets()
print(f"Train dataset size: {len(train)}")
print(f"validation dataset size: {len(validation)}")
print(f"test dataset size: {len(test)}")

Train dataset size: 174817
validation dataset size: 96811
test dataset size: 424223


In [4]:
collate_fn = lambda batch: utils.fast_collate(batch)

train_loader = torch.utils.data.DataLoader(train, batch_size=32, shuffle=False, num_workers=6, collate_fn=collate_fn, pin_memory=True)
val_loader = torch.utils.data.DataLoader(validation, batch_size=1000, shuffle=False, num_workers=6, collate_fn=collate_fn, pin_memory=True)
test_loader = torch.utils.data.DataLoader(test, batch_size=1000, shuffle=False, num_workers=6, collate_fn=collate_fn, pin_memory=True)

### Fine tunning

In [5]:
model = mobilenet(weights=pre_weights.IMAGENET1K_V2)
model.classifier[-1] = torch.nn.Linear(1280, NUM_CLASSES)

bce_loss = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(model.classifier)
print(f"is in cuda: {next(model.parameters()).is_cuda}")
print(f"device: {device}")

Sequential(
  (0): Linear(in_features=960, out_features=1280, bias=True)
  (1): Hardswish()
  (2): Dropout(p=0.2, inplace=True)
  (3): Linear(in_features=1280, out_features=1, bias=True)
)
is in cuda: True
device: cuda


In [6]:
NUM_EPOCHS = 5
for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0

    prefetcher = data_prefetcher.data_prefetcher(train_loader)
    inputs, labels = prefetcher.next()
    while inputs is not None:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        preds = model(inputs).squeeze(1)
        loss = bce_loss(preds, labels.float())

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        inputs, labels = prefetcher.next()
    
    epoch_loss = running_loss / len(train)
    print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}], Training Loss: {epoch_loss:.4f}")

    # Validation 
    model.eval()

    val_labels = []
    val_preds = []

    prefetcher = data_prefetcher.data_prefetcher(val_loader)
    inputs, labels = prefetcher.next()
    with torch.no_grad():
        while inputs is not None:
            inputs, labels = inputs.to(device), labels.to(device)
            preds = model(inputs)

            val_labels.extend(labels.cpu().numpy())
            val_preds.extend((torch.sigmoid(preds).cpu().numpy() > 0.5).astype(int))
            
            inputs, labels = prefetcher.next()


    accuracy = accuracy_score(val_labels, val_preds)
    cm = confusion_matrix(val_labels, val_preds)
    
    print(f'Validation Accuracy: {accuracy:.2f}%')
    utils.print_confusion_matrix(cm)
    print("----------------------------------------------------------------------")


Epoch [1/5], Training Loss: 0.0693


----------------------------------------------------------------------
Validation Accuracy: 0.82%
Confusion matrix:
77088 51 
32029 65649 
Epoch [2/5], Training Loss: 0.0037


----------------------------------------------------------------------
Validation Accuracy: 0.95%
Confusion matrix:
73755 3384 
5430 92248 
Epoch [3/5], Training Loss: 0.0018


----------------------------------------------------------------------
Validation Accuracy: 0.95%
Confusion matrix:
75931 1208 
7870 89808 
Epoch [4/5], Training Loss: 0.0006


----------------------------------------------------------------------
Validation Accuracy: 0.97%
Confusion matrix:
76362 777 
5057 92621 
Epoch [5/5], Training Loss: 0.0002


----------------------------------------------------------------------
Validation Accuracy: 0.96%
Confusion matrix:
76669 470 
6505 91173 


### Results

In [7]:
model.eval()

test_labels = []
test_preds = []

prefetcher = data_prefetcher.data_prefetcher(test_loader)
inputs, labels = prefetcher.next()
with torch.no_grad():
    while inputs is not None:
        inputs, labels = inputs.to(device), labels.to(device)
        preds = model(inputs)

        test_labels.extend(labels.cpu().numpy())
        test_preds.extend((torch.sigmoid(preds).cpu().numpy() > 0.5).astype(int))

        inputs, labels = prefetcher.next()

accuracy = accuracy_score(test_labels, test_preds)
cm = confusion_matrix(test_labels, test_preds)

print(f'Test Accuracy: {accuracy:.2f}%')
utils.print_confusion_matrix(cm)

Test Accuracy: 0.96%
Confusion matrix:
192894 830 
16020 214479 
